Аналитическое решение линейной регрессии (МНК)

В данном ноутбуке представлена реализация и оценка аналитического (точного) решения задачи линейной регрессии с использованием **метода наименьших квадратов (МНК)** через вычисление нормального уравнения:

$$w = (X^T X)^{-1} X^T y$$

In [11]:
import numpy as np
import pandas as pd

In [12]:
def mean_squared_error(y_true, y_pred):
    """
    Среднеквадратичная ошибка
    """
    return np.mean((y_true - y_pred) ** 2)

def root_mean_squared_error(y_true, y_pred):
    """
    Корень из среднеквадратичной ошибки 
    """
    return np.sqrt(mean_squared_error(y_true, y_pred))

def r2_score(y_true, y_pred):
    """
    Коэффициент детерминации
    """
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    
    if ss_tot == 0:
        return 0.0
        
    return 1 - (ss_res / ss_tot)

In [13]:
def standardize_features(X_train, X_test=None):
    """
    Стандартизирует признаки (Z-score нормализация)
    """
    # Считаем параметры только по обучающей выборке
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    
    # Защита от деления на ноль, если колонка константная
    std[std == 0] = 1.0
    
    # Масштабируем тренировочные данные
    X_train_scaled = (X_train - mean) / std
    
    # Если переданы тестовые данные, масштабируем их по тем же параметрам
    if X_test is not None:
        X_test_scaled = (X_test - mean) / std
        return X_train_scaled, X_test_scaled, mean, std
        
    return X_train_scaled, mean, std

In [14]:
def analytical_linear_regression(X, y):
    # w = (X^T * X)^(-1) * X^T * y
    return np.linalg.pinv(X.T @ X) @ X.T @ y

In [15]:
# Загрузка данных
df_train = pd.read_csv('../Dataset (Farhat)/dataset_sample_5000.csv')
y_train = df_train['Log_Цена'].to_numpy()
X_raw_train = df_train.drop(columns=['Log_Цена']).to_numpy()

df_test = pd.read_csv('../Dataset (Farhat)/dataset_sample_all.csv')
y_test = df_test['Log_Цена'].to_numpy()
X_raw_test = df_test.drop(columns=['Log_Цена']).to_numpy()

In [16]:
# Стандартизация
X_train_scaled, X_test_scaled, mean, std = standardize_features(X_raw_train, X_raw_test)

# Добавление константы 
X_train = np.hstack([np.ones((X_train_scaled.shape[0], 1)), X_train_scaled])
X_test = np.hstack([np.ones((X_test_scaled.shape[0], 1)), X_test_scaled])

In [17]:
# Аналитическое решение
weights_exact = analytical_linear_regression(X_train, y_train)
y_pred_exact = X_test @ weights_exact

In [18]:
# Оценка 
print("Результаты точного аналитического решения (МНК):")
print(f"MSE: {mean_squared_error(y_test, y_pred_exact):.5f}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_exact):.5f}")
print(f"R^2: {r2_score(y_test, y_pred_exact):.5f}")

Результаты точного аналитического решения (МНК):
MSE: 0.04248
RMSE: 0.20610
R^2: 0.83755
